## 0. Environment check

In [ ]:
import os
import platform
import sys
from pathlib import Path

volume_root = Path(os.environ.get("NIGHTS_WATCH_VOLUME_ROOT", "/mnt/nightswatch-poc"))
print(f"Modal volume root: {volume_root}")
print(f"Volume exists: {volume_root.exists()}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print("[DONE] Environment check complete")


## 1. Install dependencies

In [ ]:
!pip install "ultralytics==8.3.*" roboflow==1.3.3 gdown==6.0.0 pycocotools==2.0.11 kaggle==2.0.1 pandas==3.0.2 --quiet

import importlib.metadata as metadata

packages = ["ultralytics", "roboflow", "gdown", "pycocotools", "kaggle", "pandas"]
for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not found")
print("[DONE] Dependencies installed")


## 2. Clone GitHub repository

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

volume_root = Path(os.environ.get("NIGHTS_WATCH_VOLUME_ROOT", "/mnt/nightswatch-poc"))
repo_url = os.environ.get(
    "NIGHTS_WATCH_REPO_URL", "https://github.com/AdityaChaudhary2913/NightsWatch-PoC.git"
)
repo_dir = volume_root / "NightsWatch"
output_dir = volume_root / "poc_outputs"
data_dir = volume_root / "datasets"
yolo_config_dir = volume_root / ".ultralytics"
volume_root.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
yolo_config_dir.mkdir(parents=True, exist_ok=True)
os.environ["NIGHTS_WATCH_VOLUME_ROOT"] = str(volume_root)
os.environ["YOLO_CONFIG_DIR"] = str(yolo_config_dir)
if repo_dir.exists():
    print(f"Repository already exists at {repo_dir}; refreshing from remote")
    status = subprocess.run(["git", "-C", str(repo_dir), "status", "--short"], text=True, capture_output=True, check=True)
    if status.stdout.strip():
        raise RuntimeError(
            "Mounted repo has local changes; please commit/stash them or remove the repo directory before rerunning."
        )
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "HEAD"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
%cd /mnt/nightswatch-poc/NightsWatch
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
commit = subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"], text=True, capture_output=True, check=True).stdout.strip()
print(f"[DONE] Repository ready: {repo_dir}")
print(f"[DONE] Repo commit: {commit}")
print(f"[DONE] Dataset root: {data_dir}")
print(f"[DONE] Output root: {output_dir}")
print(f"[DONE] YOLO config dir: {yolo_config_dir}")


## 3. Dataset download and preparation

In [ ]:
import json
from pathlib import Path

import pandas as pd

from datasets.prepare_dronevehicle import prepare_dronevehicle
from datasets.prepare_flir import prepare_flir
from datasets.verify_dataset import save_dataset_summaries, verify_yolo_dataset
from utils.env import get_log_dir
from utils.seed import fix_all_seeds

fix_all_seeds(42)
log_dir = get_log_dir()
prep_manifest_path = log_dir / "prep_manifest.json"

def prep_manifest_ready(payload: dict) -> bool:
    required_keys = ["thermal_yaml", "eo_yaml", "ir_yaml", "pair_manifest", "dataset_records"]
    if not all(key in payload for key in required_keys):
        return False
    required_paths = [payload["thermal_yaml"], payload["eo_yaml"], payload["ir_yaml"], payload["pair_manifest"]]
    return all(Path(path).exists() for path in required_paths)

if prep_manifest_path.exists():
    prep_manifest = json.loads(prep_manifest_path.read_text(encoding="utf-8"))
else:
    prep_manifest = {}

if prep_manifest_ready(prep_manifest):
    print(f"[DONE] Existing prep manifest found at {prep_manifest_path}; skipping dataset prep")
    dataset_records = prep_manifest["dataset_records"]
else:
    if prep_manifest:
        print("[WARN] Existing prep manifest is incomplete or stale; rebuilding dataset metadata")
    dataset_records = []
    flir_result = prepare_flir()
    paired_result = prepare_dronevehicle()
    thermal_yaml = flir_result["thermal"]["yaml"]
    eo_yaml = paired_result["visible"]["yaml"]
    ir_yaml = paired_result["infrared"]["yaml"]
    pair_manifest = paired_result["pair_manifest"]
    verification_jobs = [
        ("Thermal", thermal_yaml),
        ("EO", eo_yaml),
        ("IR", ir_yaml),
    ]
    for name, yaml_path in verification_jobs:
        summary = verify_yolo_dataset(yaml_path, dataset_name=name, strict=True)
        dataset_records.append(summary)
    save_dataset_summaries(dataset_records)
    prep_manifest = {
        "thermal_yaml": thermal_yaml,
        "eo_yaml": eo_yaml,
        "ir_yaml": ir_yaml,
        "pair_manifest": pair_manifest,
        "dataset_records": dataset_records,
        "sources": {
            "thermal": flir_result["source"],
            "paired": paired_result["source"],
        },
    }
    prep_manifest_path.write_text(json.dumps(prep_manifest, indent=2), encoding="utf-8")
    print(f"[DONE] Wrote prep manifest: {prep_manifest_path}")

table = pd.DataFrame([
    {
        "dataset name": row["dataset"],
        "num train": row["train_images"],
        "num val": row["val_images"],
        "num classes": row["num_classes"],
        "image size": row["sample_image_size"],
        "source": row["source"],
    }
    for row in dataset_records
])
display(table)
print(f"[DONE] Dataset preparation complete. Pair manifest: {prep_manifest['pair_manifest']}")


## 4. Final file listing

In [ ]:
from pathlib import Path

volume_root = Path("/mnt/nightswatch-poc")
for path in sorted((volume_root / "datasets").rglob("*")):
    print(path.relative_to(volume_root))
print("[DONE] Dataset volume listing complete")
